# A5 · A result that isn't: testing with permutations

Follow `README.md` step by step. Cells marked **TODO** are yours to finish.

In [ ]:
%load_ext sql
from lakehouse import sql_engine, whoami
import math
import numpy as np
import pandas as pd
trino = sql_engine()
%sql trino --alias trino
schema = f"dbt_{whoami().lower()}"
print("your schema:", schema)

## 1. Look at the tables

In [ ]:
%%sql trino
SELECT * FROM lakehouse.samples.nfl_games LIMIT 5

In [ ]:
%%sql trino
SELECT * FROM lakehouse.samples.nfl_q4_flags LIMIT 5

## 2–3. Load the flags, define the window and "needs the cover"

We load every discretionary fourth-quarter flag with its game's spread into pandas, so the
same function can run on the real spread and on shuffled spreads (step 6).

In [ ]:
flags = pd.read_sql("""
SELECT g.game_id, g.season, g.spread, f.margin, f.home_possession, f.game_seconds_remaining,
       CASE WHEN f.penalized_side = 'defense' THEN 1 ELSE 0 END AS on_defense
FROM lakehouse.samples.nfl_q4_flags f
JOIN lakehouse.samples.nfl_games g ON g.game_id = f.game_id
WHERE f.penalty_class = 'discretionary'
""", trino)
games = pd.read_sql("""
SELECT game_id, season, spread,
       ntile(10) OVER (PARTITION BY season ORDER BY elo_diff, game_id) AS strength_decile
FROM lakehouse.samples.nfl_games
""", trino).set_index("game_id")
len(flags), len(games)

In [ ]:
def cells(flags, spread):
    """The four cells of step 3: share of flags on the defense by era and offense_needs_cover."""
    d = flags.assign(spread=np.asarray(spread))
    d["lcm"] = d.margin + d.spread                        # live cover margin
    # TODO (step 2): keep the window: last 600 s, |margin| <= 10, spread <> 0,
    #                |lcm| <= 8, lcm <> 0, and leave out season 2018.
    d = d[...]
    d["post"] = (d.season >= 2019).astype(int)
    # TODO (step 2): 1 when the team with the ball is the one not covering.
    d["needs_cover"] = ...
    return d.groupby(["post", "needs_cover"]).on_defense.agg(["mean", "size"])

table = cells(flags, flags.game_id.map(games.spread))
table

## 4–5. The difference-in-differences and its naive p-value

In [ ]:
def did_and_p(table):
    p = table["mean"]
    # TODO (step 4): (post, needs cover - post, not) - (pre, needs cover - pre, not)
    did = ...
    # TODO (step 5): se over the four cells, then p = erfc(|did/se| / sqrt(2))
    se = ...
    naive_p = ...
    return did, naive_p

observed_did, naive_p = did_and_p(table)
observed_did, naive_p

## 6. What does "no effect" look like? Shuffle the spreads

Shuffle the spreads across games **within** season × strength decile, recompute, repeat.

In [ ]:
rng = np.random.default_rng(2026)
draws = 1000
sims = []
for _ in range(draws):
    # TODO (step 6): a shuffled copy of games.spread, permuted within each
    #                (season, strength_decile) group. Hint: groupby(...).spread.transform(...)
    shuffled = ...
    sims.append(did_and_p(cells(flags, flags.game_id.map(shuffled)))[0])
sims = np.array(sims)
perm_mean = sims.mean()
perm_mean

## 7. The permutation p-value

In [ ]:
# TODO (step 7): the share of shuffles with |did| >= |observed_did|
perm_p = ...
print(f"observed {observed_did:.3f}  naive p {naive_p:.4f}  |  permutation mean {perm_mean:.3f}  permutation p {perm_p:.3f}")

## 8. Publish your five numbers

In [ ]:
results = {"observed_did": observed_did, "naive_p": naive_p, "perm_mean": perm_mean,
           "perm_p": perm_p, "draws": draws}
values = ", ".join(f"('{k}', CAST({float(v)!r} AS double))" for k, v in results.items())
statement = (f"CREATE OR REPLACE TABLE lakehouse.{schema}.a5_spread_test AS "
             f"SELECT * FROM (VALUES {values}) AS t(statistic, value)")
print(statement)

In [ ]:
%sql trino CREATE SCHEMA IF NOT EXISTS lakehouse.{{schema}}
%sql trino {{statement}}
%sql trino SELECT * FROM lakehouse.{{schema}}.a5_spread_test

## 9. Check your work

In a terminal: `lab-tracks check A5`